In [ ]:
import pandas as pd
import json
import os

print("Biblioteker lastet inn!")


In [22]:
#Last inn rådatafil

DATA_FILE = "../data/filtertreff_ukodede_diagnoser.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE, encoding="utf-8", sep=";")
    df = df.rename(columns={
        "journal_id": "id",
        "diagnosetekst": "diagnosis"
    })
    print(f"Lastet {len(df)} rader fra fil.")
else:
    print("Fant ikke filen! Sjekk at den ligger i data/-mappen.")

df.head()

Lastet 4388 rader fra fil.


,id,diagnosis
0,ID_0001,hysteria ?
1,ID_0002,hysteria 7
2,ID_0003,hysteria
3,ID_0004,202|hysteria
4,ID_0005,hysteria


In [23]:
#Datakvaliteteskontroll og overview

print(f"Antall rader: {df.shape[0]}")
print(f"Antall kolonner: {df.shape[1]}")
print(f"\nManglende verdier:\n{df.isnull().sum()}")
print(f"\nDe første radene:")
df.head()

Antall rader: 4388
Antall kolonner: 2

Manglende verdier:
id           0
diagnosis    0
dtype: int64

De første radene:


,id,diagnosis
0,ID_0001,hysteria ?
1,ID_0002,hysteria 7
2,ID_0003,hysteria
3,ID_0004,202|hysteria
4,ID_0005,hysteria


In [24]:
#RENSE DIAGNOSELISTE

# Fjern tomme diagnoser
df = df.dropna(subset=["diagnosis"])
df = df[df["diagnosis"].str.strip() != ""]

# Fjern ekstra mellomrom
df["diagnosis"] = df["diagnosis"].str.strip()
df["diagnosis"] = df["diagnosis"].str.replace(r"\s+", " ", regex=True)

# Lag en lowercase-versjon for sammenligning
df["diagnosis_clean"] = df["diagnosis"].str.lower()

# Fjern duplikater
before = len(df)
df = df.drop_duplicates(subset=["id", "diagnosis_clean"])
after = len(df)

print(f"Fjernet {before - after} tomme/duplikater.")
print(f"Gjenstående poster: {after}")
df

Fjernet 1285 tomme/duplikater.
Gjenstående poster: 3103


,id,diagnosis,diagnosis_clean
0,ID_0001,hysteria ?,hysteria ?
1,ID_0002,hysteria 7,hysteria 7
2,ID_0003,hysteria,hysteria
3,ID_0004,202|hysteria,202|hysteria
4,ID_0005,hysteria,hysteria
...,...,...,...
4383,ID_2831,psykoneurose hysteria,psykoneurose hysteria
4384,ID_2831,311 | psykoneurose hysteria,311 | psykoneurose hysteria
4385,ID_2832,hysteria,hysteria
4386,ID_2833,hysterica,hysterica


In [6]:
OUTPUT_FILE = "../data/diagnoses_clean.json"

df.to_json(OUTPUT_FILE, orient="records", force_ascii=False, indent=2)

print(f"Lagret {len(df)} rensede poster til {OUTPUT_FILE}")

Lagret 3103 rensede poster til ../data/diagnoses_clean.json
